# Automated Unit Test Generator and Code Translator

This notebook provides:
1. **Unit test generation** — Generate pytest (Python) or Jest-style (JavaScript) unit tests for your code using an LLM.
2. **Code translation** — Translate code between **Python**, **JavaScript**, and **TypeScript**.

**Input:** Code snippet + mode (generate tests / translate) + target language (for translation).  
**Output:** Generated unit tests or translated code, with basic validation.

## Setup and dependencies

In [ ]:
# Uncomment and run if needed:
# !pip install gradio openai python-dotenv

In [1]:
import os
import re
import ast
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

load_dotenv(override=True)

True

## Model configuration (open-source LLM via Ollama)

Uses **Ollama** (open-source, local) by default. Optionally use **OpenAI** if `OPENAI_API_KEY` is set.

In [2]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"
OLLAMA_MODEL = "llama3.2"  # or codellama, deepseek-coder, etc.
OPENAI_MODEL = "gpt-4o-mini"  # used only if OPENAI_API_KEY is set

ollama_client = OpenAI(api_key="ollama", base_url=OLLAMA_BASE_URL)
openai_client = None
if os.getenv("OPENAI_API_KEY"):
    openai_client = OpenAI()
    print("OpenAI client available (OPENAI_API_KEY set).")
else:
    print("Using Ollama only. Set OPENAI_API_KEY to also use OpenAI.")

def get_client(use_openai: bool):
    if use_openai and openai_client:
        return openai_client, OPENAI_MODEL
    return ollama_client, OLLAMA_MODEL

OpenAI client available (OPENAI_API_KEY set).


## Unit test generation and code translation logic

In [4]:
UNIT_TEST_SYSTEM = """You are an expert at writing unit tests. Generate only the test code, no explanations.
- For Python: use pytest. Import the code under test if needed (assume it's in the same module or suggest a short import). Use assert statements or pytest.assert_*.
- For JavaScript: use Jest or a simple test runner style (describe/it, expect). Use clear test names and cover normal and edge cases.
Output ONLY valid code inside a single markdown code block. No preamble."""

TRANSLATE_SYSTEM = """You are an expert at translating code between programming languages. Preserve behavior and idioms.
Supported: Python, JavaScript, TypeScript.
Output ONLY the translated code inside a single markdown code block. No explanation. Use correct syntax for the target language."""

def extract_code_block(text: str) -> str:
    match = re.search(r"```(?:\w+)?\s*\n([\s\S]*?)```", text)
    if match:
        return match.group(1).strip()
    return text.strip()

In [5]:
def call_llm(client, model: str, system: str, user: str) -> str:
    try:
        resp = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": user},
            ],
            temperature=0.3,
        )
        return (resp.choices[0].message.content or "").strip()
    except Exception as e:
        return f"Error calling LLM: {e}"

def generate_unit_tests(code: str, lang: str, use_openai: bool) -> str:
    client, model = get_client(use_openai)
    framework = "pytest" if lang == "Python" else "Jest"
    user = f"Generate {framework} unit tests for this {lang} code. Cover main behavior and edge cases.\n\n```{lang.lower()}\n{code}\n```"
    raw = call_llm(client, model, UNIT_TEST_SYSTEM, user)
    return extract_code_block(raw)

def translate_code(code: str, source_lang: str, target_lang: str, use_openai: bool) -> str:
    client, model = get_client(use_openai)
    user = f"Translate this {source_lang} code to {target_lang}. Preserve logic and style.\n\n```{source_lang}\n{code}\n```"
    raw = call_llm(client, model, TRANSLATE_SYSTEM, user)
    return extract_code_block(raw)

## Basic output validation

In [6]:
def validate_python(code: str) -> tuple[bool, str]:
    try:
        ast.parse(code)
        return True, "Valid Python syntax."
    except SyntaxError as e:
        return False, f"Syntax error: {e}"

def validate_js_ts(code: str) -> tuple[bool, str]:
    # Basic checks: balanced braces, no obvious issues
    if not code.strip():
        return False, "Empty code."
    open_b = code.count("{") - code.count("}")
    if open_b != 0:
        return False, f"Unbalanced braces (difference: {open_b})."
    return True, "Basic structure OK (brace-balanced)."

def validate_output(code: str, lang: str) -> str:
    if lang == "Python":
        ok, msg = validate_python(code)
    else:
        ok, msg = validate_js_ts(code)
    status = "OK" if ok else "Warning"
    return f"Validation ({lang}): [{status}] {msg}"

## Gradio UI

In [7]:
MODES = ["Generate unit tests", "Translate code"]
LANGUAGES = ["Python", "JavaScript", "TypeScript"]

def run(code: str, mode: str, source_lang: str, target_lang: str, use_openai: bool) -> str:
    if not (code and code.strip()):
        return "Please enter some code."
    use_openai = bool(use_openai)
    if mode == "Generate unit tests":
        out = generate_unit_tests(code, source_lang, use_openai)
        validation = validate_output(out, source_lang)
        return f"{out}\n\n---\n{validation}"
    else:
        if source_lang == target_lang:
            return "For translation, please choose a different target language than source."
        out = translate_code(code, source_lang, target_lang, use_openai)
        validation = validate_output(out, target_lang)
        return f"{out}\n\n---\n{validation}"

In [8]:
with gr.Blocks(title="Unit Test Generator & Code Translator", theme=gr.themes.Soft()) as demo:
    gr.Markdown("### Unit Test Generator & Code Translator")
    gr.Markdown("*Generate unit tests*: set **Source/code language** to the language of your code. *Translate*: set source and target.")
    code_in = gr.Textbox(
        label="Paste your code",
        placeholder="def add(a, b): return a + b",
        lines=12,
    )
    mode = gr.Radio(MODES, value="Generate unit tests", label="Mode")
    with gr.Row():
        source_lang = gr.Dropdown(LANGUAGES, value="Python", label="Source / code language")
        target_lang = gr.Dropdown(LANGUAGES, value="JavaScript", label="Target language (for translation)")
    use_openai = gr.Checkbox(False, label="Use OpenAI (if API key set)")
    btn = gr.Button("Run")
    out = gr.Textbox(label="Output", lines=20)

    btn.click(
        fn=run,
        inputs=[code_in, mode, source_lang, target_lang, use_openai],
        outputs=out,
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


## Usage notes

- **Ollama**: Ensure Ollama is running (`ollama serve`) and a model is pulled (e.g. `ollama pull llama3.2`).
- **Generate unit tests**: Set *Source/code language* to the language of your pasted code; tests are generated in that language (pytest for Python, Jest-style for JavaScript).
- **Translate**: Set *Source* to the language of your code and *Target* to the desired output language.
- **Validation**: Python output is checked with `ast.parse`; JS/TS get a simple brace-balance check. For stronger validation, run the generated tests (e.g. save and run `pytest` for Python).

In [ ]:
# Optional: validate generated Python tests by saving and running pytest
# (Run this after generating tests; replace generated_tests with the output from the app.)

# import tempfile
# import subprocess
# generated_tests = """
# def test_example():
#     assert 1 + 1 == 2
# """
# with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False) as f:
#     f.write(generated_tests)
#     path = f.name
# result = subprocess.run(["pytest", path, "-v"], capture_output=True, text=True)
# print(result.stdout or result.stderr)